In [1]:
pip install ipykernel


In [2]:
%pip install -U langchain-core langchain-community langchain-ollama faiss-cpu sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install -U langchain-qdrant qdrant-client

Note: you may need to restart the kernel to use updated packages.


In [4]:
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models


# Load your merged JSON
with open("soft_corner_rag_corpus.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to LangChain Documents
documents = [
    Document(
        page_content=item["page_content"],
        metadata=item["metadata"]
    )
    for item in data
]

# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# Create vector store
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embedding_model,
    url="http://localhost:6333", 
    collection_name="soft_corner_docs",
    force_recreate=True # Ensures a fresh start in the Docker container
)

print("✅ Embedding + Qdrant DB created in Docker!")

C:\Users\win 11\AppData\Local\Temp\ipykernel_13948\171751498.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
d:\AI_CHATBOT_FINAL\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11719.39it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ign

✅ Embedding + Qdrant DB created in Docker!


In [5]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [7]:

import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models


# Load your merged JSON
with open("soft_corner_rag_corpus.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Convert to LangChain Documents
documents = [
    Document(
        page_content=item["page_content"],
        metadata=item["metadata"]
    )
    for item in data
]

# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# Create vector store
vectorstore = QdrantVectorStore.from_documents(
    documents=documents,
    embedding=embedding_model,
    url="http://localhost:6333", 
    collection_name="soft_corner_docs",
    force_recreate=True # Ensures a fresh start in the Docker container
)

print("✅ Embedding + Qdrant DB created in Docker!")




import os
from dotenv import load_dotenv
from groq import Groq
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_core.prompts import PromptTemplate

load_dotenv()
# Load embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

# 1. Load Qdrant DB (pointing to the folder you created earlier)

client_qdrant = QdrantClient(url="http://localhost:6333")

vectorstore = QdrantVectorStore(
    client=client_qdrant,
    collection_name="soft_corner_docs",
    embedding=embedding_model,
)

# 2. Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# Use the variable in your client setup
client_groq = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# 4. Prompt template
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are the official assistant for the Soft Corner app.

Soft Corner is a private and secure space designed for singles, couples, and married partners to express feelings, store memories, and build meaningful relationships.

Your role is to:
- Help users understand how Soft Corner works
- Explain features clearly and accurately
- Guide users step-by-step when needed
- Maintain a warm, supportive, and trustworthy tone

STRICT RULES:
- Answer ONLY using the provided context
- Do NOT make up information
- If the answer is not in the context, say:
  "I don't know based on the available information."
- Do NOT assume features that are not mentioned
- Keep answers clear, natural, and helpful

TONE:
- Warm, supportive, and relationship-focused
- Simple and easy to understand
- Slightly emotional but not exaggerated
- Professional and trustworthy

STYLE:
- Use short paragraphs
- Use bullet points when explaining features
- Highlight key points clearly

---

Context:
{context}

---

User Question:
{question}

---

Answer:
"""
)

# 5. Run the RAG Chain
query = "what is Maximum Privacy?"

# Retrieve context from Qdrant
results = retriever.invoke(query)
context = "\n\n".join([res.page_content for res in results])

# Format and Send to Groq
final_prompt = prompt.format(context=context, question=query)

response = client_groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": final_prompt}],
    temperature=0
)

print("\nLLM Answer :\n")
print(response.choices[0].message.content)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8026.72it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding + Qdrant DB created in Docker!


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9648.21it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



LLM Answer :

Maximum Privacy is a feature in Soft Corner that ensures your space is completely secure. This means that your memories and feelings are protected with multiple layers of protection. 

The key benefits of Maximum Privacy include:
* Mandatory passcode every time you access your space
* Screenshot-proof to prevent unauthorized sharing
* No sharing or downloading of your content
* Your space is hidden from the main app for extra security

This feature gives you peace of mind, knowing that your private moments and feelings are safe and secure.


In [ ]:
import os
from typing import List, Dict, Any

from dotenv import load_dotenv
from groq import Groq
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings


load_dotenv()


# -----------------------------
# Configuration
# -----------------------------
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "soft_corner_docs")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "BAAI/bge-small-en-v1.5")

TOP_K = int(os.getenv("TOP_K", "6"))
MIN_SCORE = float(os.getenv("MIN_SCORE", "0.0"))  # set > 0 if you want to filter weak hits


# -----------------------------
# Validation
# -----------------------------
def validate_env() -> None:
    if not GROQ_API_KEY:
        raise ValueError("Missing GROQ_API_KEY in environment variables.")


# -----------------------------
# Clients
# -----------------------------
def build_embedding_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)


def build_qdrant_client() -> QdrantClient:
    return QdrantClient(url=QDRANT_URL)


def build_groq_client() -> Groq:
    return Groq(api_key=GROQ_API_KEY)


# -----------------------------
# Retrieval
# -----------------------------
def extract_text_from_payload(payload: Dict[str, Any]) -> str:
    """
    Adjust these keys to match however you stored data in Qdrant.
    Common choices are: text, page_content, content, chunk_text.
    """
    if not payload:
        return ""

    for key in ("page_content", "text", "content", "chunk_text"):
        value = payload.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    return ""


def retrieve_context(
    query: str,
    qdrant_client: QdrantClient,
    embedding_model: HuggingFaceEmbeddings,
    collection_name: str,
    top_k: int = 6,
    min_score: float = 0.0,
) -> List[Dict[str, Any]]:
    """
    Direct Qdrant retrieval.
    Returns a list of dicts containing text, score, and payload.
    """
    query_vector = embedding_model.embed_query(query)

    response = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )

    retrieved_docs: List[Dict[str, Any]] = []

    for point in response.points:
        payload = point.payload or {}
        text = extract_text_from_payload(payload)
        score = getattr(point, "score", None)

        if not text:
            continue

        if score is not None and score < min_score:
            continue

        retrieved_docs.append(
            {
                "id": getattr(point, "id", None),
                "score": score,
                "text": text,
                "payload": payload,
            }
        )

    return retrieved_docs


def build_context(docs: List[Dict[str, Any]]) -> str:
    """
    Creates the final context block sent to the LLM.
    Keeps source structure visible for better grounding.
    """
    chunks = []

    for i, doc in enumerate(docs, start=1):
        source = (
            doc["payload"].get("source")
            or doc["payload"].get("file_name")
            or doc["payload"].get("document_name")
            or "unknown_source"
        )

        chunks.append(
            f"[Document {i}]"
            f"\nSource: {source}"
            f"\nScore: {doc['score']}"
            f"\nContent:\n{doc['text']}"
        )

    return "\n\n".join(chunks)


# -----------------------------
# Generation
# -----------------------------
SYSTEM_PROMPT = """
You are the official assistant for the Soft Corner app.

Soft Corner is a private and secure space designed for singles, couples, and married partners
to express feelings, store memories, and build meaningful relationships.

Your role is to:
- Help users understand how Soft Corner works
- Explain features clearly and accurately
- Guide users step-by-step when needed
- Maintain a warm, supportive, and trustworthy tone

STRICT RULES:
- Answer ONLY using the provided context
- Do NOT make up information
- If the answer is not in the context, say:
  "I don't know based on the available information."
- Do NOT assume features that are not mentioned
- Keep answers clear, natural, and helpful

TONE:
- Warm, supportive, and relationship-focused
- Simple and easy to understand
- Slightly emotional but not exaggerated
- Professional and trustworthy

STYLE:
- Use short paragraphs
- Use bullet points when explaining features
- Highlight key points clearly
""".strip()


def generate_answer(
    groq_client: Groq,
    model_name: str,
    context: str,
    question: str,
) -> str:
    user_prompt = f"""
Context:
{context}

User Question:
{question}

Answer:
""".strip()

    response = groq_client.chat.completions.create(
        model=model_name,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content.strip()


# -----------------------------
# Main pipeline
# -----------------------------
def ask_soft_corner(question: str) -> str:
    validate_env()

    embedding_model = build_embedding_model()
    qdrant_client = build_qdrant_client()
    groq_client = build_groq_client()

    docs = retrieve_context(
        query=question,
        qdrant_client=qdrant_client,
        embedding_model=embedding_model,
        collection_name=QDRANT_COLLECTION,
        top_k=TOP_K,
        min_score=MIN_SCORE,
    )

    if not docs:
        return "I don't know based on the available information."

    context = build_context(docs)
    answer = generate_answer(
        groq_client=groq_client,
        model_name=GROQ_MODEL,
        context=context,
        question=question,
    )

    return answer


if __name__ == "__main__":
    query = "what is Maximum Privacy?"
    answer = ask_soft_corner(query)

    print("\nLLM Answer:\n")
    print(answer)